In [1]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY is not set. Add it to your .env file.")

genai.configure(api_key=gemini_api_key)
gemini_client = genai.GenerativeModel("gemini-2.5-flash")

print("API client successfully configured")
print(f"Loaded GEMINI_API_KEY: {gemini_api_key[:5]}...")

API client successfully configured
Loaded GEMINI_API_KEY: AIzaS...


d:\UTE\LLM-agentic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\PC\AppData\Local\Temp\ipykernel_18156\2951529727.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
from IPython.display import display, Markdown

def print_markdown(text):
    display(Markdown(text))

In [3]:
def get_ai_tutor_response(user_question):
    """
    Sends a question to the Gemini API, asking it to respond as an AI Tutor.
    Args:
        user_question (str): The question asked by the user
    Returns:
        str: The AI's response, or an error message
    """
    system_prompt = "You are a helpful and patient AI Tutor. Explain concepts clearly and concisely."

    try:
        response = gemini_client.generate_content(
            f"{system_prompt}\n\nStudent question: {user_question}",
            generation_config=genai.types.GenerationConfig(
                temperature=0.7
            )
        )
        return response.text
    except Exception as e:
        return f"Sorry, I encountered an error trying to get an answer: {e}"


In [4]:
test_question = "Could you explain the concept of functions in Python and their purpose in programming?"

tutor_answer = get_ai_tutor_response(test_question)

print_markdown(f"""
### Asking the AI Tutor

{test_question}

### AI Tutor's Response

{tutor_answer}
""")


### Asking the AI Tutor

Could you explain the concept of functions in Python and their purpose in programming?

### AI Tutor's Response

Absolutely! Let's break down functions in Python.

### What is a Function?

Imagine a function as a **mini-program** or a **recipe** that you can define once and then use multiple times. It's a named block of code designed to perform a specific task.

### Purpose of Functions in Programming

Functions are incredibly important because they help you:

1.  **Avoid Repetition (DRY Principle):** Instead of writing the same lines of code over and over again for a task, you write it once inside a function. This is known as the "Don't Repeat Yourself" (DRY) principle.
2.  **Organize Code:** Functions break down large programs into smaller, manageable, and logical chunks. Each chunk (function) handles a specific part of the overall problem.
3.  **Improve Readability:** Well-named functions make your code easier to understand, as the function name often describes what it does.
4.  **Promote Reusability:** Once a function is written, you can call it from different parts of your program, or even in different programs, without having to rewrite its internal logic.
5.  **Easier Debugging:** If there's an issue, you can focus on debugging a smaller, isolated function rather than an entire program.

### How to Define and Use a Function in Python

Here's a quick look at the basic structure:

```python
# 1. Define a function
def greet(name): # 'def' keyword, function name, parameters in parentheses
    """This function greets the person passed in as a parameter.""" # Optional docstring
    print(f"Hello, {name}!") # Indented block of code (the function's body)

# 2. Call/Use the function
greet("Alice") # Pass an argument to the function
greet("Bob")
```

**Explanation:**

*   `def`: This keyword tells Python you are defining a new function.
*   `greet`: This is the name of the function. Choose descriptive names!
*   `(name)`: This is a **parameter**. It's a placeholder for information the function needs to do its job. When you call the function, you pass an **argument** (like `"Alice"`) which gets assigned to `name`.
*   `:`: A colon marks the end of the function header.
*   Indentation: The lines of code that belong to the function must be indented (usually 4 spaces).
*   `print(...)`: This is the task the function performs.
*   `greet("Alice")`: This is how you **call** or **invoke** the function. Python executes the code inside the `greet` function, using `"Alice"` as the value for `name`.

Think of functions as powerful tools to make your code cleaner, more efficient, and easier to manage!


In [5]:
import gradio as gr

In [6]:
ai_tutor_interface_simple = gr.Interface(
    fn = get_ai_tutor_response,
    inputs = gr.Textbox(lines = 2, placeholder= "Ask the AI Tutor anything...", label= "Your Question"),
    outputs= gr.Textbox(label = "AI Tutor's Answer"),
    title= "🤖 Simple AI Tutor",
    description= "Enter your question below and the AI Tutor will provide an explanation. Powered by GeminiAI.",
    flagging_mode = "never",
)
print("Launching Gradio Interface...")
ai_tutor_interface_simple.launch()

Launching Gradio Interface...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [7]:
def stream_ai_tutor_response(user_question):
    """
    Sends a question to the GeminiAI API and streams the response as a generator.
    Args:
        user_question (str): The question asked by the user.
    Yields:
        str: Chunks of the AI's response. 
    """
    system_prompt = "You are a helpful and patient AI Tutor. Explain concepts clearly and concisely."

    try:
        stream = gemini_client.generate_content(
            f"{system_prompt}\n\nStudent question: {user_question}",
            generation_config=genai.types.GenerationConfig(
                temperature=0.7
            ),
            stream=True
        )
        full_response = ""
        for chunk in stream:
            if chunk.text:
                text_chunk = chunk.text
                full_response += text_chunk
                yield full_response
    
    except Exception as e:
        yield f"Sorry, I encountered an error: {e}"


In [8]:
ai_tutor_interface_streaming = gr.Interface(
    fn = stream_ai_tutor_response,
    inputs = gr.Textbox(lines = 2, placeholder= "Ask the AI Tutor anything...", label= "Your Question"),
    outputs= gr.Markdown(
        label = "AI Tutor's Answer (Streaming)", container= True, height= 250
    ),
    title= "🤖 AI Tutor with Streaming",
    description= "Enter your question. The answer will appear word-by-word!",
    flagging_mode = "never",
)
print("Launching Streaming Gradio Interface...")
ai_tutor_interface_streaming.launch()

Launching Streaming Gradio Interface...
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [9]:
explanation_levels= {
    1: "like I'm 5 years old",
    2: "Like I'm 10 years old",
    3: "like a high school student",
    4: "like a college student",
    5: "like an expert in the field"
}

In [10]:
def stream_ai_tutor_response_with_level(user_question, explanation_level_value):
    """
    Streams AI Tutor response based on user question and selected explanation level.
    Args:
        user_question (str): The question from the user.
        explanation_level_value (int): The value from the slider (1-5).
    
    Yields:
        str: Chunks of the AI's response.
    """
    level_description = explanation_levels.get(
        explanation_level_value, "clearly and concisely"
    )
    system_prompt = f"You are a helpful AI Tutor. Explain the following concept {level_description}"
    print(f"DEBUG: Using System Prompt: '{system_prompt}'")

    try:
        stream = gemini_client.generate_content(
            f"{system_prompt}\n\nStudent question: {user_question}",
            generation_config=genai.types.GenerationConfig(
                temperature=0.7
            ),
            stream=True
        )
        full_response = ""

        for chunk in stream:
            if chunk.text:
                text_chunk = chunk.text
                full_response += text_chunk
                yield full_response
    except Exception as e:
        print(f"An error occurred during streaming: {e}")
        yield f"Sorry, I encountered an error: {e}"

In [11]:
ai_tutor_interface_slider = gr.Interface(
    fn= stream_ai_tutor_response_with_level,
    inputs=[
        gr.Textbox(lines = 3, placeholder= "Ask the AI Tutor a question...", label = "Your Question"),
        gr.Slider(
            minimum= 1,
            maximum= 5,
            step = 1,
            value= 3, #default level (high school)
            label= "Explanation Level",
        ),
    ],
    outputs= gr.Markdown(label="AI Tutor's Explanation (Streaming)", container= True, height= 250),
    title= "🎓 Advanced AI Tutor",
    description= "Ask a question and select and desired level of explanation using the slider.",
    flagging_mode = "never",
)
print("Launching Advanced Interface with Slider...")
ai_tutor_interface_slider.launch()

Launching Advanced Interface with Slider...
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


DEBUG: Using System Prompt: 'You are a helpful AI Tutor. Explain the following concept like a high school student'
